# 04 — Exploration des stratégies : du plancher trivial au modèle appris

**Projet** : Résumé de comptes-rendus d'intervention — encodeur-décodeur appris, baseline extractive publiée

## Objectifs pédagogiques

1. Mesurer les planchers triviaux : résumé vide, recopie intégrale, premières phrases. Un score n'a de sens que comparé à ce que le hasard et la paresse obtiennent.
1. Comparer les deux baselines extractives de la stack sur **les mêmes lignes** de validation.
1. Entraîner un encodeur-décodeur minuscule sur le corpus réduit et le mesurer avec exactement les mêmes métriques.
1. Lire ce que le modèle regarde (parts d'attention par phrase) et ce qu'il ne peut pas savoir (taux de jetons inconnus du vocabulaire).

## 0. Environnement et corpus

Le corpus réduit des notebooks (`__NB_DOCUMENTS__` comptes-rendus) et une architecture réduite gardent l'exécution de ce notebook en quelques secondes. Les valeurs absolues ne sont **pas** celles de la référence publiée par le README.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (200 comptes-rendus), un budget
# d'époques réduit et une architecture réduite : l'exécution reste de quelques secondes, et les
# valeurs absolues d'un notebook ne sont **pas** celles de la référence publiée par le README.
NB_DOCUMENTS = 200
NB_EPOCHS = 8

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "model.params.layers=1",
                "model.params.units=96",
                "model.params.pretraining.epochs=1",
                "+model.params.vocab_size=800",
                "+model.params.min_frequency=2",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
                "train.early_stopping.enabled=false",
                "seed=42",
                "log_level=WARNING",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Bac à sable : les notebooks génèrent **leur** corpus (réduit) et écrivent **leurs** artefacts sous
# `outputs/notebooks/`. `data/raw`, `artifacts/` et donc les chiffres publiés par le README restent
# ceux de `make data` / `make train` : aucun notebook ne peut les modifier par accident.
NB_ROOT = PROJECT_ROOT / "outputs" / "notebooks"
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=NB_ROOT / "data",
    artifacts_dir=NB_ROOT / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
ID_COLUMN = str(MODEL_NODE.get("id_column", "doc_id"))
COMPRESSION = float(MODEL_NODE.get("params", {}).get("compression", 0.45))
MAX_OUTPUT_TOKENS = int(MODEL_NODE.get("params", {}).get("max_output_tokens", 90))
STRATEGIES = list(MODEL_NODE.get("strategies_to_compare", []) or ["lead"])

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.45)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Stratégies comparées : {', '.join(STRATEGIES)}")
print(f"Budget de longueur: compression {COMPRESSION:.2f}, plafond {MAX_OUTPUT_TOKENS} tokens")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticSummaryCorpusGenerator
from src.data.loaders import SummaryCorpusLoader

LOADER = SummaryCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS, REFERENCES, FACTS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(
        f"Corpus du notebook lu : {len(DOCUMENTS)} comptes-rendus, "
        f"{len(REFERENCES)} résumés de référence, {len(FACTS)} faits annotés"
    )
else:
    # Un clone frais n'a pas encore de corpus réduit : le notebook le génère, de façon déterministe,
    # dans son bac à sable. `make data` produit le corpus de référence (600 comptes-rendus) dans
    # `data/raw` — les deux ne se mélangent jamais.
    bundle = SyntheticSummaryCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, REFERENCES, FACTS = bundle.documents, bundle.queries, bundle.facts
    METADATA = bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_references(REFERENCES)
    LOADER.save_facts(FACTS)
    LOADER.save_metadata(METADATA)
    print(f"Corpus réduit généré dans outputs/notebooks/data : {len(DOCUMENTS)} comptes-rendus")

TRAIN_DOCUMENTS, TRAIN_REFERENCES, TRAIN_FACTS = LOADER.load_split("train")
VAL_DOCUMENTS, VAL_REFERENCES, VAL_FACTS = LOADER.load_split("val")
TEST_DOCUMENTS, TEST_REFERENCES, TEST_FACTS = LOADER.load_split("test")
SPLIT_SIZES = LOADER.split_sizes()
# Le test ne sert **jamais** à régler quoi que ce soit : les réglages se lisent sur la validation,
# le verdict sur le test, une seule fois.
GOLDEN = dict(zip(REFERENCES["doc_id"], REFERENCES["summary"], strict=False))
print("splits :", SPLIT_SIZES)
print("compression moyenne demandée :", round(float(REFERENCES["compression"].mean()), 4))

## 1. Les planchers triviaux

Trois systèmes que personne n'appellerait « un modèle », et qu'aucun modèle n'a le droit de ne pas battre : un résumé vide, la recopie intégrale du document, et les premières phrases. Le troisième est publié comme **référence** dans le rapport d'évaluation.

In [ ]:
from src.evaluation.evaluator import SummaryEvaluator
from src.evaluation.rouge import corpus_rouge
from src.models import build_model
from src.models.lead import LeadSummarizer
from src.training.metrics import trivial_floor

val_references = VAL_REFERENCES["summary"].tolist()
empty_rouge = corpus_rouge(val_references, [""] * len(val_references))
copy_rouge = corpus_rouge(val_references, VAL_DOCUMENTS["text"].tolist())

# L'évaluateur est le même objet que celui du pipeline `evaluate` : scorer une stratégie ici, c'est
# appliquer exactement les métriques publiées (ROUGE **et** fidélité), sur les mêmes lignes.
SCORER = SummaryEvaluator(
    LeadSummarizer(compression=COMPRESSION, max_output_tokens=MAX_OUTPUT_TOKENS, seed=42),
    config=CONFIG.model_dump(),
    metrics_config=CONFIG.metrics.model_dump(),
    paths=NB_PATHS,
)
lead = build_model(CONFIG, algorithm="lead")
lead.fit(TRAIN_DOCUMENTS, TRAIN_REFERENCES)
lead_scored = SCORER.score_model(lead, VAL_DOCUMENTS, VAL_REFERENCES, VAL_FACTS)

FLOOR_METRICS = ("rouge1_f", "rouge2_f", "rouge_l_f", "fact_coverage", "compression")


def mean_scores(frame: pd.DataFrame) -> dict[str, float]:
    """Moyenne des métriques publiées d'une table de prédictions scorée."""
    return {name: round(float(frame[name].mean()), 4) for name in FLOOR_METRICS}


floors = pd.DataFrame(
    [
        {"stratégie": "resume_vide", **trivial_floor(), "compression": 0.0},
        {
            "stratégie": "recopie_integrale",
            **copy_rouge,
            "fact_coverage": float("nan"),
            "compression": 1.0,
        },
        {"stratégie": "lead", **mean_scores(lead_scored)},
    ]
)
display(floors.set_index("stratégie").round(4))
print("planchers mesurés sur", len(VAL_DOCUMENTS), "documents de validation")
print("contenu des références mesuré par la copie intégrale :", round(copy_rouge["rouge1_f"], 4))

**Ce qu'il faut retenir**

- Un système muet obtient 0,0 : la métrique est *définie* sur une sortie vide, ce qui est la condition pour publier un verdict chiffré sur un autre système.
- La recopie intégrale obtient un ROUGE non nul et une couverture nulle : elle contient tout, donc rien de vérifiable. C'est le piège que la couverture des faits existe pour lever.
- Le plancher ``lead`` n'est pas ridicule — il prend les phrases du début, qui portent souvent l'équipement et le symptôme. C'est pour cela qu'il est la référence publiée : battre un plancher faible ne prouve rien.

## 2. Ce que la stack sait produire

Trois stratégies, deux familles : extractive (on sélectionne des phrases du document, donc on ne peut pas halluciner une valeur, mais on ne peut pas résumer) et apprise (on génère, donc on doit être mesuré sur sa fidélité).

In [ ]:
from src.models import available_algorithms, describe_algorithm

rows = []
for name in available_algorithms():
    spec = describe_algorithm(name)
    rows.append(
        {
            "algorithme": name,
            "famille": spec["family"],
            "paramètres appris": spec["requires_fit"],
            "ce qu'il fait": spec["notes"],
        }
    )
display(pd.DataFrame(rows))

### 2.1 Deux baselines extractives sur la même validation

``lead`` prend les premières phrases, ``textrank`` sélectionne les phrases les plus centrales puis les diversifie (MMR). Toutes deux écrivent un résumé **fidèle par construction** : chaque mot vient du document.

In [ ]:
from src.training.metrics import per_strategy_frame

SERVED_ALGORITHM = str(CONFIG.model.algorithm)
# Les stratégies comparées sont **celles de la configuration** : le notebook n'invente pas sa
# propre liste, sinon la comparaison publiée ne serait pas celle que `mode=all` reproduit.
EXTRACTIVE = [name for name in STRATEGIES if name != SERVED_ALGORITHM]
print("stratégies comparées :", ", ".join([SERVED_ALGORITHM, *EXTRACTIVE]))

SCORED = [lead_scored]
for algorithm in EXTRACTIVE:
    model = build_model(CONFIG, algorithm=algorithm)
    model.fit(TRAIN_DOCUMENTS, TRAIN_REFERENCES, validation=(VAL_DOCUMENTS, VAL_REFERENCES))
    SCORED.append(SCORER.score_model(model, VAL_DOCUMENTS, VAL_REFERENCES, VAL_FACTS))

extractives = pd.concat(SCORED, ignore_index=True)
display(per_strategy_frame(extractives).round(4))
print("compression moyenne :", round(float(extractives["compression"].mean()), 4))

**Ce qu'il faut retenir**

- ``textrank`` sélectionne les phrases centrales mais les coupe pour tenir le budget : il produit des résumés plus courts que ``lead``, et le ROUGE le sanctionne — la couverture des faits, elle, reste comparable.
- Chaque baseline est ajustée sur le **train** et jugée sur la **validation** : c'est ce qui rend la comparaison lisible, et c'est la règle que suit le pipeline d'évaluation (`mode=evaluate`).
- Aucune des deux ne peut inventer une valeur : leur fiabilité de 1,0 est structurelle, pas une performance. Le modèle appris, lui, devra la mériter.

### 2.2 Un encodeur-décodeur entraîné ici

Vocabulaire WordPiece partagé appris sur le train, pré-entraînement par débruitage, puis affinage supervisé. Le notebook ne fait qu'appeler le modèle de la stack : il n'y a pas de seconde implémentation à maintenir.

In [ ]:
import time

from src.models.summarizer import EncoderDecoderSummarizer

learned = build_model(CONFIG, algorithm="transformer_tiny")
started = time.perf_counter()
fit_metrics = learned.fit(
    TRAIN_DOCUMENTS, TRAIN_REFERENCES, validation=(VAL_DOCUMENTS, VAL_REFERENCES)
)
duration = time.perf_counter() - started
print(
    f"ajustement en {duration:.1f}s :", {key: round(value, 4) for key, value in fit_metrics.items()}
)
print(learned.summary())

In [ ]:
transformer_scored = SCORER.score_model(learned, VAL_DOCUMENTS, VAL_REFERENCES, VAL_FACTS)
EVERYTHING = pd.concat([extractives, transformer_scored], ignore_index=True)
comparison = per_strategy_frame(EVERYTHING)
display(comparison.round(4))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
comparison.set_index("strategy")["rouge1_f"].plot.bar(
    ax=axes[0], rot=20, title="ROUGE-1 par stratégie (validation)"
)
comparison.set_index("strategy")["fact_coverage"].plot.bar(
    ax=axes[1], rot=20, title="Couverture des faits (validation)"
)
plt.tight_layout()
plt.show()

print(
    "part des résumés qui atteignent leur budget (modèle appris) :",
    round(float(transformer_scored["hit_max_length"].mean()), 4),
)

**Ce qu'il faut retenir**

- Le budget d'entraînement d'un notebook est volontairement minuscule : l'objectif ici est de vérifier que **la mécanique** tourne et que les métriques répondent, pas d'atteindre le seuil contractuel, qui est mesuré par `mode=all`.
- La part des résumés qui atteignent leur budget se lit avec le ROUGE : un décodeur qui bute sur sa borne ne conclut pas sa phrase, et un score bas peut venir de là plutôt que du modèle.
- Un modèle appris qui obtient un meilleur ROUGE mais une couverture plus basse a appris à ressembler à la référence **sans** être vrai : les deux colonnes sont publiées côte à côte pour que ce cas soit visible.

## 3. Ce que le modèle regarde

Un encodeur-décodeur n'est pas une boîte noire : on peut lire la répartition de son attention sur les phrases du document au premier pas de décodage, et savoir sur quels jetons son vocabulaire est aveugle.

In [ ]:
from src.preprocessing.transformers import split_sentences

example_row = TEST_DOCUMENTS.iloc[0]
example_id, example_text = str(example_row["doc_id"]), str(example_row["text"])

if isinstance(learned, EncoderDecoderSummarizer):
    shares = learned.attention_shares(example_text)
    document_sentences = split_sentences(example_text)
    for index, share in shares.items():
        if index.isdigit() and int(index) < len(document_sentences):
            print(f"phrase {index:>2s} : {share:.3f} — {document_sentences[int(index)][:90]}")
    print()
    print("attention totale répartie sur", len(shares), "phrase(s)")

card = learned.model_card()
print("vocabulaire appris     :", card["vocabulary"]["size"], "pièces")
print("jetons inconnus (train):", card["vocabulary"]["unknown_rate"])
print("paramètres             :", card["n_parameters"])
print("décodage               :", card["architecture"]["decoding"])

**Ce qu'il faut retenir**

- Une part d'attention élevée sur la phrase qui porte l'équipement ou la durée est une explication vérifiable : elle se compare aux faits saillants, pas à l'intuition de l'analyste.
- Le taux de jetons inconnus est la mesure du vocabulaire : un vocabulaire trop petit transforme les références de pièces en suites de caractères, et le modèle ne peut plus les recopier à l'identique.
- Le décodage glouton est déclaré dans la fiche modèle : un résumé doit être reproductible, donc aucune température ni aucun échantillonnage n'est employé.